## Objective:
To build a synthetic market environment (a return matrix $R$) that flawlessly obeys the laws of mathematical physics. Randomly generating a covariance matrix ($\Sigma$) will result in negative variance (impossible kinetic energy) and asymmetrical correlations, causing the eigendecomposition engine to crash. We must physically construct a strictly Positive Semi-Definite (PSD) matrix.

**The Physics Analogy:** 

The Fleet & The WindImagine a fleet of $N$ ships. We must program the baseline wind patterns ($L$) and the individual shaking of the ships' hulls ($D$) to generate a mathematically valid Tangle Map ($\Sigma$).

**Mathematical Execution:**

We utilize the Cholesky-inspired architectural cheat code: $\Sigma = LL^T + D$
1. Factor Loadings ($L$): A randomly generated matrix representing uncalibrated, chaotic wind forces.
2. Squaring the Physics ($LL^T$): By multiplying $L$ by its transpose, we execute the matrix equivalent of squaring a number.
$$\forall x \in \mathbb{R}^N, \quad x^T(LL^T)x = \vert{}\vert{}L^Tx\vert{}\vert{}^2 \ge 0$$
This proof guarantees all variance in the system is strictly positive or zero.
Idiosyncratic Variance ($D$): A diagonal matrix of strictly positive values added to ensure no asset possesses exactly zero variance, preventing matrix singularity.

**The Output:**

We map a realistic daily expected return ($\mu$) and pull $T$ days of simulated returns from a multivariate normal distribution governed by our PSD covariance matrix.

$$R \sim \mathcal{N}(\mu, \Sigma) \implies R \in \mathbb{R}^{T \times N}$$

In [19]:
import numpy as np
from pathlib import Path

np.random.seed(19)

### I. Covariance matrix generation

In [20]:
def generate_cov(N: int) -> np.ndarray:
    L = np.random.uniform(0.5, 1.5, size=(N, 1))
    
    D_variances = np.random.uniform(0.1, 0.5, size=N)
    D = np.diag(D_variances)
    
    cov = L @ L.T + D
    
    assert np.allclose(cov, cov.T), "Covariance matrix is assymetric"
    
    return cov

cov = generate_cov(5)
cov

array([[0.49024623, 0.75363909, 0.44632054, 0.38130513, 0.49681726],
       [0.75363909, 1.95954168, 0.94207531, 0.80484341, 1.04866174],
       [0.44632054, 0.94207531, 0.98055386, 0.47664479, 0.62103901],
       [0.38130513, 0.80484341, 0.47664479, 0.90030882, 0.5305724 ],
       [0.49681726, 1.04866174, 0.62103901, 0.5305724 , 1.04556768]])

### II. Mu matrix generation 

In [21]:
def generate_mu(N: int) -> np.ndarray:
    return np.random.uniform(-0.03, 0.03, size=N)

mu = generate_mu(5)
mu

array([-0.0170446 ,  0.00294165,  0.0027336 , -0.01595544, -0.02317645])

### III. Return matrix generation 

In [22]:
def generate_R(T: int, mu: np.ndarray, cov: np.ndarray) -> np.ndarray:
    return np.random.multivariate_normal(mu, cov, T)

generate_R(1000, mu, cov)

array([[ 0.36700757,  2.11190751, -0.30649273,  0.67646971,  1.73716215],
       [-0.64968151, -2.25331201, -0.62840674,  0.06506461, -2.3710406 ],
       [ 0.65507595,  0.95767017,  0.27966901,  1.16090112,  0.71275218],
       ...,
       [-0.39022918,  0.41628376, -0.69339131, -0.32790789,  0.26874994],
       [ 1.0040787 ,  1.13299345, -0.12816716, -0.0996392 , -0.4671146 ],
       [-0.28381117, -0.08078616, -0.91606736, -0.15289367, -1.05216205]],
      shape=(1000, 5))

### IV. Synthetic data generation

In [ ]:
def generate_synthetic_data(T: int, N: int, seed: int=19) -> np.ndarray:
    np.random.seed(seed)
    
    cov = generate_cov(N)
    mu = generate_mu(N)
    
    if T <= 0 or N <= 0:
        raise ValueError(f'Matrix Size should be a positive non zero value')
    
    elif cov.shape != (N, N):
        raise ValueError(f'Covariance matrix shape mismatch. Expected ({N}, {N})')
    
    elif mu.shape[0] != N:
        raise ValueError(f'Mu matrix shape mismatch. Expected ({N}, )')
    
    R = generate_R(T, mu, cov)
    
    if R.shape == (T, N): 
        current_dir = Path.cwd()
        
        target_dir = current_dir.parent / 'data'
        
        target_dir.mkdir(parents=True, exist_ok=True)
        
        file_path = target_dir / f'synthetic_market_{seed}.npy'
        
        np.save(file_path, R)
        return R
    
    raise ValueError(f'R matrix output shape mismatch. Expected ({T}, {N})')

generate_synthetic_data(T=1000, N=5, seed=13)

array([[ 1.82890063,  0.77317112,  0.95642841,  1.91909303,  1.74018709],
       [ 1.05554011,  1.20134157,  1.44596338,  1.32488673,  1.91929634],
       [-0.88736892,  0.32618939,  0.50358246, -1.89311083,  0.72957991],
       ...,
       [ 1.96206711,  2.00891639,  2.50437218,  3.05973581,  4.48503151],
       [-1.3466122 , -1.47369962, -0.5818216 , -0.65078649, -1.3476751 ],
       [ 1.55995006,  0.12353328, -0.21102493,  0.02917614,  0.4970604 ]],
      shape=(1000, 5))